# Welfare dynamics and final-hub ranking

In [ ]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import scienceplots  # noqa: F401
from IPython.display import display
from joblib import Parallel, delayed, effective_n_jobs
from matplotlib.ticker import MaxNLocator
from matplotlib_inline.backend_inline import set_matplotlib_formats
from tqdm.auto import tqdm

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
extra = 3
plt.rcParams.update({
    "font.size": 14+extra,
    "axes.titlesize": 17+extra,
    "axes.labelsize": 15+extra,
    "xtick.labelsize": 12+extra,
    "ytick.labelsize": 12+extra,
    "legend.fontsize": 12+extra,
    "legend.title_fontsize": 13+extra,
    "figure.titlesize": 19+extra,
})

project_root = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "pyproject.toml").is_file() and (path / "tasks").is_dir()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Run this notebook from the network-creation repository.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from network_creation.core.model import ActorScheduler
from network_creation.core.observability import ObservationSampling
from network_creation.io import read_json
from network_creation.observers.hub_ranking_observer import (
    HubRankingObserver,
    HubRankingResult,
)
from network_creation.observers.social_welfare_observer import (
    CenteredMeanWelfareObserver,
)
from network_creation.observers.state_history_observer import StateHistoryObserver
from network_creation.plotting.graph_history import build_circle_layout
from network_creation.stats.hub_ranking import (
    simultaneous_cmc_band,
    tie_aware_hit_at_k,
)
from network_creation.time_normalization import normalized_source_times
from tasks.policies import (
    load_policy_results,
    prepare_instrumented_policy_simulation,
)

## Configuration

In [ ]:
CONFIG = {
    "source_zarr": (
        project_root
        / "artifacts"
        / "simulation_runs"
        / "run-20260802T181747.zarr"
    ),
    "group_size": 100,
    "objective": "centered_mean_welfare.mean",
    "visualization": {
        "seed": 20260804,
        "n_episodes": 32,
        "checkpoint_every": 1.0,
        "episode": 0,
        "graph_times": [0, 10, 20, 30],
        "graph_side": 2.0,
    },
    "ranking": {
        "group_sizes": [10, 50, 100],
        "seed": 13,
        "n_episodes": 2**10,
        "n_jobs": 30,
        "bootstrap_seed": 13,
        "bootstrap_resamples": 4000,
    },
}

## Select the final-welfare argmax

In [ ]:
def select_best_input(config, group_size):
    result_store = Path(config["source_zarr"])
    group_size = int(group_size)
    objective = str(config["objective"])
    results = load_policy_results(
        result_store, stats=(objective,), normalize_time=False
    )
    final_step = int(
        results.coords["last_valid_raw_step"].sel(n_agents=group_size).item()
    )
    scores = results["x"].sel(
        n_agents=group_size, observable=objective, step=final_step, drop=True
    )
    values = np.asarray(scores.values, dtype=float)
    best_welfare = float(np.nanmax(values))
    positions = np.argwhere(np.isfinite(values) & (values == best_welfare))
    candidates = []
    for position in positions:
        selector = {
            dimension: scores.coords[dimension].values[int(index)].item()
            for dimension, index in zip(scores.dims, position, strict=True)
        }
        input_index = int(
            results.coords["input_index"].sel(
                {"n_agents": group_size, **selector}
            ).item()
        )
        candidates.append((input_index, selector))
    input_index, selector = min(candidates, key=lambda candidate: candidate[0])
    theta_row = results["theta"].sel(
        {"n_agents": group_size, **selector}, drop=True
    )
    theta = {
        str(name): float(value)
        for name, value in zip(
            theta_row.coords["theta_parameter"].values,
            theta_row.values,
            strict=True,
        )
    }
    config_path = result_store / "run_config.json"
    if not config_path.is_file():
        config_path = Path(results.attrs["config_path"])
    run_config = read_json(config_path)
    return {
        "run_config": run_config,
        "group_size": group_size,
        "input_index": input_index,
        "theta": theta,
    }

In [ ]:
selection = select_best_input(CONFIG, CONFIG["group_size"])
n_agents = selection["group_size"]
n_actors = min(
    int(selection["run_config"]["simulator"]["actor_scheduler"]["n_actors"]),
    n_agents,
)
raw_checkpoint_interval = int(round(
    CONFIG["visualization"]["checkpoint_every"] * n_agents / n_actors
))
figure_output_dir = (
    project_root / "artifacts" / "figures" / Path(CONFIG["source_zarr"]).stem
)
figure_output_dir.mkdir(parents=True, exist_ok=True)


def run_selected(selected_input, options, observers, *, description):
    run_config = copy.deepcopy(selected_input["run_config"])
    run_config["simulator"]["seed"] = int(options["seed"])
    run_config["simulator"]["n_episodes"] = int(options["n_episodes"])
    prepared = prepare_instrumented_policy_simulation(
        run_config,
        input_index=selected_input["input_index"],
        observers=observers,
    )
    for episode in tqdm(
        range(prepared.settings.n_episodes),
        desc=description,
        unit="episode",
    ):
        prepared.simulator.run(
            n_episodes=1, reset_observers=(episode == 0)
        )
    return prepared

## Welfare and graph snapshots

In [ ]:
def plot_simulation_summary(
    welfare_observer, history_observer, *, episode, times, graph_side
):
    welfare_result = welfare_observer.get()
    welfare_runs = np.stack(welfare_result.outputs["runs"], axis=0)
    welfare_times = normalized_source_times(
        welfare_result.steps, n_agents=n_agents, n_actors=n_actors
    ).reshape(-1)
    history = history_observer.get()
    graph_steps = history.runs_steps[episode]
    graph_times = normalized_source_times(
        graph_steps, n_agents=n_agents, n_actors=n_actors
    ).reshape(-1)
    graph_indices = [int(np.abs(graph_times - time).argmin()) for time in times]
    snapshots = [
        (float(graph_times[index]), history.runs_graphs[episode][index])
        for index in graph_indices
    ]

    series_height = 3.0
    figure = plt.figure(
        figsize=(graph_side * len(snapshots), series_height + graph_side + 0.5),
        layout="compressed",
    )
    grid = figure.add_gridspec(
        2, len(snapshots), height_ratios=(series_height, graph_side)
    )
    axis = figure.add_subplot(grid[0, :])
    for values in welfare_runs:
        axis.plot(welfare_times, values, color="#1f4e79", alpha=0.1, linewidth=1.2)
    welfare_mean = welfare_runs.mean(axis=0)
    welfare_std = welfare_runs.std(axis=0)
    axis.fill_between(
        welfare_times, welfare_mean - welfare_std, welfare_mean + welfare_std,
        color="#1f4e79", alpha=0.25, label="mean ± SD",
    )
    axis.plot(
        welfare_times, welfare_mean, color="#1f4e79", linewidth=2.5,
        label="mean normalized welfare",
    )
    axis.set(
        xlim=(welfare_times[0], welfare_times[-1]),
        xlabel="activation opportunities per agent",
        ylabel="normalized welfare",
        title="Normalized welfare across rerun episodes",
    )
    axis.grid(alpha=0.3)
    axis.legend()

    positions = build_circle_layout(n_agents)
    for column, (time, graph) in enumerate(snapshots):
        axis = figure.add_subplot(grid[1, column])
        graph_nx = graph.to_networkx(vertex_attr_hashable="name")
        nx.draw_networkx_edges(
            graph_nx, pos=positions, width=0.45, edge_color="#1f4e79",
            alpha=0.28, arrows=False, ax=axis,
        )
        nx.draw_networkx_nodes(
            graph_nx, pos=positions, node_size=18, node_color="#1f4e79",
            edgecolors="#18354d", linewidths=0.35, ax=axis,
        )
        axis.set(xlim=(-1.08, 1.08), ylim=(-1.08, 1.08), aspect="equal")
        axis.set_axis_off()
        axis.text(
            0.5, -0.04, f"Time {time:g}", transform=axis.transAxes,
            ha="center", va="top",
        )
    return figure

In [ ]:
visualization = CONFIG["visualization"]
sampling = ObservationSampling.periodic(raw_checkpoint_interval)
welfare_observer = CenteredMeanWelfareObserver(
    n_agents=n_agents,
    observation_sampling=sampling,
    benefit_to_cost_ratio=selection["theta"]["benefit_to_cost_ratio"],
    distance_discount=selection["theta"]["distance_discount"],
    aggregations=("runs",),
)
history_observer = StateHistoryObserver(
    observation_sampling=sampling, keep_runs=True
)
visualization_run = run_selected(
    selection,
    visualization,
    (welfare_observer, history_observer),
    description=f"Visualization (n={n_agents})",
)
summary_figure = plot_simulation_summary(
    welfare_observer,
    history_observer,
    episode=int(visualization["episode"]),
    times=visualization["graph_times"],
    graph_side=float(visualization["graph_side"]),
)
summary_figure_path = (
    figure_output_dir / "welfare_dynamics_and_network_snapshots.pdf"
)
summary_figure.savefig(
    summary_figure_path, format="pdf", bbox_inches="tight"
)
display(summary_figure)
plt.close(summary_figure)

## Rank distribution of the final-network hub

In [ ]:
def run_ranking_chunk(
    chunk_index, selected_input, options, n_episodes, simulation_seed
):
    observer = HubRankingObserver(n_agents=selected_input["group_size"])
    run_config = copy.deepcopy(selected_input["run_config"])
    run_config["simulator"]["seed"] = int(options["seed"])
    run_config["simulator"]["n_episodes"] = int(n_episodes)
    prepared = prepare_instrumented_policy_simulation(
        run_config,
        input_index=selected_input["input_index"],
        observers=(observer,),
    )
    prepared.simulator.rng = np.random.default_rng(simulation_seed)
    prepared.simulator.actor_scheduler = ActorScheduler(
        seed=simulation_seed,
        n_agents=prepared.settings.n_agents,
        n_actors=prepared.settings.n_actors,
        activation_weights=tuple(
            agent.activation_weight for agent in prepared.simulator.agents
        ),
    )
    prepared.simulator.run(n_episodes=n_episodes, reset_observers=True)
    return chunk_index, observer.get()


def run_ranking_parallel(parallel, selected_input, options):
    n_episodes = int(options["n_episodes"])
    if n_episodes < 1:
        raise ValueError("n_episodes must be positive")
    n_chunks = min(
        n_episodes, 4 * effective_n_jobs(int(options["n_jobs"]))
    )
    base_size, remainder = divmod(n_episodes, n_chunks)
    chunk_sizes = [
        base_size + (chunk_index < remainder)
        for chunk_index in range(n_chunks)
    ]
    seed_sequences = np.random.SeedSequence(int(options["seed"])).spawn(
        len(chunk_sizes)
    )
    chunk_seeds = [
        int(sequence.generate_state(1, dtype=np.uint64)[0])
        for sequence in seed_sequences
    ]
    jobs = (
        delayed(run_ranking_chunk)(
            chunk_index, selected_input, options, chunk_size, chunk_seed
        )
        for chunk_index, (chunk_size, chunk_seed) in enumerate(
            zip(chunk_sizes, chunk_seeds, strict=True)
        )
    )
    chunks = [None] * len(chunk_sizes)
    completed = parallel(jobs)
    for chunk_index, result in tqdm(
        completed,
        total=len(chunk_sizes),
        desc=f"Ranking (n={selected_input['group_size']})",
        unit="chunk",
    ):
        chunks[chunk_index] = result
    return HubRankingResult(
        actor_order=np.concatenate([chunk.actor_order for chunk in chunks]),
        recipient_order=np.concatenate(
            [chunk.recipient_order for chunk in chunks]
        ),
        final_hub_id=np.concatenate([chunk.final_hub_id for chunk in chunks]),
    )


def plot_hub_cmc(results_by_group_size, options):
    names = ("First to act", "First to receive a link")
    colors = ("#1f4e79", "#d97706")
    figure, axes = plt.subplots(
        1, len(results_by_group_size), figsize=(15.5, 6.5),
        sharey=True, layout="constrained",
    )
    figure.suptitle("Rank distribution of the final-network hub")
    for column, (group_size, result) in enumerate(results_by_group_size.items()):
        eligible = result.final_hub_id >= 0
        hub_ids = result.final_hub_id[eligible]
        scores = np.stack(
            (
                tie_aware_hit_at_k(result.actor_order[eligible], hub_ids),
                tie_aware_hit_at_k(result.recipient_order[eligible], hub_ids),
            ),
            axis=1,
        )
        band = simultaneous_cmc_band(
            scores,
            n_resamples=int(options["bootstrap_resamples"]),
            seed=int(options["bootstrap_seed"]) + column,
        )
        ranks = np.arange(1, group_size + 1)
        chance = ranks / group_size
        axis = axes[column]
        for index, (name, color) in enumerate(zip(names, colors, strict=True)):
            axis.fill_between(
                ranks, band.low[index], band.high[index],
                color=color, alpha=0.16,
            )
            axis.plot(
                ranks, band.observed[index], color=color, linewidth=2.2,
                label=name,
            )
        axis.plot(
            ranks, chance, color="#555555", linestyle="--", linewidth=1.6,
            label="Chance level",
        )
        ticks = np.linspace(0.0, 1.0, 6)
        axis.set_yticks(ticks, [rf"{100 * tick:.0f}\%" for tick in ticks])
        axis.set(
            xlim=(1, group_size), ylim=(0.0, 1.0),
            xlabel=r"Rank threshold, $k$",
            title=rf"Network size ($n = {group_size}$)",
        )
        axis.set_box_aspect(1)
        axis.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))
        axis.minorticks_off()
        axis.grid(alpha=0.25)
        if column == 0:
            axis.set_ylabel(
                r"Cumulative probability, $\Pr(R_{\mathrm{hub}} \leq k)$"
            )
    handles, labels = axes[0].get_legend_handles_labels()
    figure.legend(
        handles, labels, loc="outside lower center", ncols=len(labels)
    )
    return figure

In [ ]:
ranking = CONFIG["ranking"]
ranking_results = {}
with Parallel(
    n_jobs=int(ranking["n_jobs"]),
    backend="loky",
    return_as="generator_unordered",
    batch_size=1,
) as parallel:
    for column, group_size in enumerate(ranking["group_sizes"]):
        ranking_selection = select_best_input(CONFIG, group_size)
        run_options = {**ranking, "seed": int(ranking["seed"]) + column}
        ranking_results[group_size] = run_ranking_parallel(
            parallel, ranking_selection, run_options
        )
cmc_figure = plot_hub_cmc(ranking_results, ranking)
cmc_figure_path = figure_output_dir / "final_hub_cmc_by_group_size.pdf"
cmc_figure.savefig(cmc_figure_path, format="pdf", bbox_inches="tight")
display(cmc_figure)
plt.close(cmc_figure)